In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,2,103.230375
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,2,235.033443
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,2,37.789691
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,2,311.534525
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,2,159.454061
...,...,...,...,...,...,...,...,...,...,...,...
53995,person_time,impairment,anemia,severe,95_plus,severe,1,baseline,0,9,0.000000
53996,person_time,impairment,anemia,severe,95_plus,severe,2,baseline,0,9,0.000000
53997,person_time,impairment,anemia,severe,95_plus,severe,3,baseline,0,9,0.000000
53998,person_time,impairment,anemia,severe,95_plus,severe,4,baseline,0,9,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
zero            10
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

not_anemic    13500
mild          13500
moderate      13500
severe        13500
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  4.946147e+06
              2                  4.080288e+06
              3                  3.640604e+06
              4                  3.483123e+06
              5                  3.383978e+06
intervention  1                  4.946147e+06
              2                  4.080288e+06
              3                  3.640604e+06
              4                  3.483123e+06
              5                  3.383978e+06
zero          1                  4.946102e+06
              2                  4.080276e+06
              3                  3.640592e+06
              4                  3.483116e+06
              5                  3.383972e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.636168e+06
              2                  2.087857e+06
              3                  1.767116e+06
              4                  1.567866e+06
              5                  1.325377e+06
intervention  1                  2.636168e+06
              2                  2.087857e+06
              3                  1.767116e+06
              4                  1.567866e+06
              5                  1.325377e+06
zero          1                  2.825492e+06
              2                  2.227686e+06
              3                  1.882413e+06
              4                  1.665643e+06
              5                  1.379191e+06
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.532974
              2                  0.511694
              3                  0.485391
              4                  0.450132
              5                  0.391662
intervention  1                  0.532974
              2                  0.511694
              3                  0.485391
              4                  0.450132
              5                  0.391662
zero          1                  0.571256
              2                  0.545965
              3                  0.517062
              4                  0.478205
              5                  0.407566
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,48678.711021
1,Female,0.0,0.019178,not_pregnant,2,42723.422230
2,Female,0.0,0.019178,not_pregnant,3,37932.716572
3,Female,0.0,0.019178,not_pregnant,4,35728.167677
4,Female,0.0,0.019178,not_pregnant,5,28631.476092
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,18328.880607
281,Male,95.0,125.000000,not_pregnant,2,19140.234171
282,Male,95.0,125.000000,not_pregnant,3,19770.250303
283,Male,95.0,125.000000,not_pregnant,4,20851.307187


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    4.478246e+06
2    3.707329e+06
3    3.317141e+06
4    3.157155e+06
5    3.080667e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.386789e+06
              2                  1.897016e+06
              3                  1.610110e+06
              4                  1.421137e+06
              5                  1.206581e+06
intervention  1                  2.386789e+06
              2                  1.897016e+06
              3                  1.610110e+06
              4                  1.421137e+06
              5                  1.206581e+06
zero          1                  2.558226e+06
              2                  2.024070e+06
              3                  1.715169e+06
              4                  1.509767e+06
              5                  1.255574e+06
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,2,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,2,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,2,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,2,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
26995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,baseline,0,9,0.0
26996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,baseline,0,9,0.0
26997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,baseline,0,9,0.0
26998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,baseline,0,9,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders', 'maternal_disorders_to_recovered_from_maternal_disorders'], dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.938625e+06
              2                  1.735531e+06
              3                  2.208574e+06
              4                  1.979411e+06
              5                  1.250466e+06
intervention  1                  2.938625e+06
              2                  1.735531e+06
              3                  2.208574e+06
              4                  1.979411e+06
              5                  1.250466e+06
zero          1                  3.021682e+06
              2                  1.781845e+06
              3                  2.254118e+06
              4                  2.020434e+06
              5                  1.264893e+06
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_scenario below, so we'd need to
# change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_deaths = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention,0,2,7710.278999
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention,0,2,5156.249081
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention,0,2,4722.545887
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention,0,2,4915.302862
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention,0,2,4626.167400
...,...,...,...,...,...,...,...,...,...,...,...,...
1195,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,6,771.027900
1196,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,6,578.270925
1197,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,6,481.892437
1198,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,6,385.513950


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  193142.488935
              2                  159024.504363
              3                  141146.294933
              4                  129147.173240
              5                  126304.007859
intervention  1                  193142.488935
              2                  159024.504363
              3                  141146.294933
              4                  129147.173240
              5                  126304.007859
zero          1                  193528.002885
              2                  159361.829069
              3                  141724.565858
              4                  129388.119459
              5                  126400.386347
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,39601.769161,zero
1,Female,0.0,0.019178,2,32639.498130,zero
2,Female,0.0,0.019178,3,28574.791970,zero
3,Female,0.0,0.019178,4,25160.080603,zero
4,Female,0.0,0.019178,5,18957.590376,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,8666.102516,intervention
746,Male,95.0,125.000000,2,8217.835776,intervention
747,Male,95.0,125.000000,3,8273.215336,intervention
748,Male,95.0,125.000000,4,8136.541097,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.224695e+08
              2                  1.150335e+08
              3                  1.138088e+08
              4                  1.086529e+08
              5                  1.031774e+08
intervention  1                  1.224695e+08
              2                  1.150335e+08
              3                  1.138088e+08
              4                  1.086529e+08
              5                  1.031774e+08
zero          1                  1.291309e+08
              2                  1.213341e+08
              3                  1.193293e+08
              4                  1.134219e+08
              5                  1.058060e+08
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.248563e+08
              2                  1.169305e+08
              3                  1.154189e+08
              4                  1.100741e+08
              5                  1.043840e+08
intervention  1                  1.248563e+08
              2                  1.169305e+08
              3                  1.154189e+08
              4                  1.100741e+08
              5                  1.043840e+08
zero          1                  1.316891e+08
              2                  1.233581e+08
              3                  1.210445e+08
              4                  1.149317e+08
              5                  1.070616e+08
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  5246.953936
              2                  4635.136215
              3                  4161.262947
              4                  3923.865884
              5                  3341.791859
baseline      1                  4876.706136
              2                  4356.155225
              3                  3949.450442
              4                  3750.256772
              5                  3275.586326
intervention  1                  2765.654300
              2                  2658.591985
              3                  2580.494909
              4                  2577.118449
              5                  2743.104885
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)